# Une fusion ciblée récupère l'essentiel du gain

Ce carnet rejoue la section 7.6 et la section 8.2 : ce que gagne une
fusion à deux termes entre l'intensité et la cohérence, et ce que perd la
disjonction.

Table employée : `exp43_fusion_ciblee_coherence.csv`, qui porte les quatre
règles de fusion sur trois lectures — l'emprise commune aux huit produits, la
fusion des douze, et l'emprise propre à chaque paire.

In [ ]:
import os
import pandas as pd

# les carnets se lancent depuis notebooks/ ; les tables sont dans data/results
RESULTATS = os.path.join("..", "data", "results")
TABLES = os.path.join("..", "tables")

def lire(dossier, nom, **kw):
    """Lit une table du depot. Le separateur et la virgule decimale varient
    d'une campagne a l'autre : on laisse pandas les deviner sauf indication."""
    kw.setdefault("sep", None)
    kw.setdefault("engine", "python")
    return pd.read_csv(os.path.join(dossier, nom), **kw)

def presque(a, b, tol=0.0006):
    """Egalite au millieme pres, tolerance par defaut d'un demi-millieme."""
    assert abs(a - b) < tol, "%.4f attendu, %.4f obtenu" % (b, a)
    return True


Les lectures disponibles dans la table.

In [ ]:
e = lire(RESULTATS, "exp43_fusion_ciblee_coherence.csv")
for lecture in sorted(e.lecture.unique()):
    print("%-40s %d lignes" % (lecture, (e.lecture == lecture).sum()))

Les quatre règles de fusion, sur l'emprise commune.

In [ ]:
huit = e[e.lecture == "A - emprise des huit"].sort_values("moyenne",
                                                          ascending=False)
print(huit[["score", "auc_ems", "auc_chatmap", "moyenne"]].to_string(index=False))

Le « environ 73 % du gain » de la page d'accueil, recalculé.

In [ ]:
# le gain de la fusion a deux termes, et sa part du gain a douze termes
tstat = huit.loc[huit.score == "notre T-stat seul", "moyenne"].iloc[0]
deux = huit.loc[huit.score == "fusion moyenne_ponderee", "moyenne"].iloc[0]
douze = (e[e.lecture == "A - emprise des huit (fusion des 12)"]
         .query("score == 'fusion moyenne_ponderee'")["moyenne"].iloc[0])
print("T-stat seul              : %.4f" % tstat)
print("fusion a deux termes     : %.4f  (gain %+.4f)" % (deux, deux - tstat))
print("fusion des douze         : %.4f  (gain %+.4f)" % (douze, douze - tstat))
part = (deux - tstat) / (douze - tstat)
print("\npart du gain a douze termes recuperee par deux termes : %.1f %%"
      % (100 * part))
presque(part, 0.73, tol=0.02)

La disjonction dégrade le classement dans chaque configuration.

In [ ]:
# la disjonction : retenir le plus alarmant des deux rangs
for lecture in [l for l in e.lecture.unique() if l.startswith("B")]:
    bloc = e[e.lecture == lecture]
    seul = bloc[bloc.score == "notre T-stat seul"]["moyenne"]
    disj = bloc[bloc.score == "fusion maximum"]["moyenne"]
    if len(seul) and len(disj):
        print("%-42s T-stat %.4f   disjonction %.4f   ecart %+.4f"
              % (lecture, seul.iloc[0], disj.iloc[0],
                 disj.iloc[0] - seul.iloc[0]))